In [1]:
import os
import csv
import time
from pathlib import Path

def memory_snapshot(write_csv=True, out_path=None):
    """
    One-shot system + per-process memory snapshot for Jupyter.
    Returns per-process data, /proc/meminfo, and optional CSV path.
    """

    def read_meminfo():
        info = {}
        with open("/proc/meminfo") as f:
            for line in f:
                k, v = line.split(":", 1)
                info[k] = int(v.strip().split()[0])  # kB
        return info

    def read_smaps_rollup(pid):
        path = f"/proc/{pid}/smaps_rollup"
        data = {}
        try:
            with open(path) as f:
                for line in f:
                    k, v = line.split(":", 1)
                    data[k] = int(v.strip().split()[0])  # kB
            return data
        except (FileNotFoundError, PermissionError, ProcessLookupError):
            return None

    def get_comm(pid):
        try:
            with open(f"/proc/{pid}/comm") as f:
                return f.read().strip()
        except Exception:
            return "?"

    meminfo = read_meminfo()
    processes = []

    for pid in filter(str.isdigit, os.listdir("/proc")):
        smaps = read_smaps_rollup(pid)
        if not smaps:
            continue

        processes.append({
            "pid": int(pid),
            "comm": get_comm(pid),
            "rss_kb": smaps.get("Rss", 0),
            "pss_kb": smaps.get("Pss", 0),
            "swap_kb": smaps.get("Swap", 0),
            "anon_kb": smaps.get("Anonymous", 0),
            "file_kb": smaps.get("File", 0),
            "shared_clean_kb": smaps.get("Shared_Clean", 0),
            "shared_dirty_kb": smaps.get("Shared_Dirty", 0),
        })

    csv_path = None
    if write_csv and processes:
        if out_path is None:
            out_path = Path(f"memory_snapshot_{int(time.time())}.csv")
        else:
            out_path = Path(out_path)

        with open(out_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=processes[0].keys())
            writer.writeheader()
            writer.writerows(processes)

        csv_path = out_path

    return processes, meminfo, csv_path


In [2]:
procs, meminfo, csv_path = memory_snapshot()
import pandas as pd

df = pd.DataFrame(procs)
display(df.sort_values("pss_kb", ascending=False).head(15))
display(df[df["swap_kb"] > 0].sort_values("swap_kb", ascending=False).head(10))
total_pss_mb = df["pss_kb"].sum() / 1024
used_mem_mb = (meminfo["MemTotal"] - meminfo["MemAvailable"]) / 1024

print(total_pss_mb, used_mem_mb)


,pid,comm,rss_kb,pss_kb,swap_kb,anon_kb,file_kb,shared_clean_kb,shared_dirty_kb
113,1466587,chrome,417540,323450,16332,280036,0,104756,7152
17,1036,chrome,277440,203722,99904,145864,0,87716,4512
112,1466276,jupyter-lab,170772,163629,0,150512,0,8568,0
18,1037,brave,118444,98039,97476,57132,0,30324,324
104,1461164,chrome,133484,95086,41324,94108,0,37012,5196
9,997,Xorg,92344,87189,70480,83200,0,5892,1052
60,5405,chrome,134160,81349,54336,50428,0,70792,1804
122,1466931,chrome,113640,77982,14512,35572,0,41688,956
108,1461374,chrome,104544,71733,17440,69892,0,29632,6504
102,1458966,chrome,130080,71626,19824,66952,0,59724,6716


,pid,comm,rss_kb,pss_kb,swap_kb,anon_kb,file_kb,shared_clean_kb,shared_dirty_kb
86,9359,code,4840,2737,285220,2260,0,2440,0
78,8318,code,22856,19666,116928,18932,0,3776,4
73,7119,code,38420,34782,111056,30880,0,4052,884
17,1036,chrome,277440,203722,99904,145864,0,87716,4512
18,1037,brave,118444,98039,97476,57132,0,30324,324
64,5662,brave,48832,30395,77172,17132,0,27092,220
9,997,Xorg,92344,87189,70480,83200,0,5892,1052
58,4654,chromium,9432,7130,69776,6664,0,2688,80
14,1032,chromium,47844,37169,61792,18704,0,16352,108
91,10245,brave,17176,8539,58216,8756,0,8288,1880


2286.7255859375 14284.83203125


,pid,comm,rss_kb,pss_kb,swap_kb,anon_kb,file_kb,shared_clean_kb,shared_dirty_kb
114,10589,steamwebhelper,408292,383593,1293552,351548,0,25572,5760
159,837578,dota2,2256344,2248546,728564,2209964,0,3848,5316
126,12278,code,17812,10419,339976,9140,0,8364,0
106,9943,code,7764,3575,320364,3080,0,4544,0
91,8103,code,37188,22003,198132,18540,0,17688,12
17,863,chrome,249924,174557,162736,122752,0,85232,3152
123,11381,chromium,214992,160330,140844,154504,0,59968,4096
94,8349,Discord,445176,425013,133124,392224,0,27840,3144
14,859,chromium,171596,134306,122904,113152,0,46684,688
95,8416,steam,38732,31404,118272,20704,0,2096,7408


In [3]:
procs, meminfo, csv_path = memory_snapshot()
import pandas as pd

df = pd.DataFrame(procs)
# display(df.sort_values("pss_kb", ascending=False).head(15))
# display(df[df["swap_kb"] > 0].sort_values("swap_kb", ascending=False).head(10))
total_pss_mb = df["pss_kb"].sum() / 1024
used_mem_mb = (meminfo["MemTotal"] - meminfo["MemAvailable"]) / 1024

# print(total_pss_mb, used_mem_mb)


# ===== Configuration =====
TOP_N = 10          # how many processes to list
KB = 1024

# Helper to format GiB nicely
def gib(kb):
    return f"{kb / KB / KB:.2f} GiB"

# ---------- Top PSS consumers ----------
print("Proportional Set Size\n")

top_pss = (
    df.groupby("comm", as_index=False)
      .agg(pss_kb=("pss_kb", "sum"))
      .sort_values("pss_kb", ascending=False)
      .head(TOP_N)
)

for _, row in top_pss.iterrows():
    print(f"{row['comm']:<16} ~{gib(row['pss_kb'])} PSS")

# print("\nTotal: "
#       f"~{gib(top_pss['pss_kb'].sum())} of real memory.\n")

# ---------- Top swap offenders ----------
print("Swap:\n")

top_swap = (
    df[df["swap_kb"] > 0]
      .groupby("comm", as_index=False)
      .agg(swap_kb=("swap_kb", "sum"))
      .sort_values("swap_kb", ascending=False)
      .head(TOP_N)
)

for _, row in top_swap.iterrows():
    print(f"{row['comm']:<16} ~{gib(row['swap_kb'])} swap")

# ---------- Sanity totals ----------
total_pss = df["pss_kb"].sum()
total_swap = df["swap_kb"].sum()

print("\n--- Totals ---")
print(f"Total PSS accounted: {gib(total_pss)}")
print(f"Total swap in use:   {gib(total_swap)}")

used_mem = meminfo["MemTotal"] - meminfo["MemAvailable"]
print(f"Kernel-reported used memory: {gib(used_mem)}")


Proportional Set Size

chrome           ~1.07 GiB PSS
code             ~0.26 GiB PSS
python           ~0.24 GiB PSS
brave            ~0.19 GiB PSS
jupyter-lab      ~0.16 GiB PSS
Xorg             ~0.08 GiB PSS
chromium         ~0.08 GiB PSS
gnome-system-mo  ~0.06 GiB PSS
corectrl         ~0.05 GiB PSS
pavucontrol      ~0.03 GiB PSS
Swap:

code             ~0.63 GiB swap
brave            ~0.58 GiB swap
chrome           ~0.55 GiB swap
chromium         ~0.32 GiB swap
Xorg             ~0.07 GiB swap
pavucontrol      ~0.03 GiB swap
gnome-system-mo  ~0.02 GiB swap
terminator       ~0.02 GiB swap
i3               ~0.02 GiB swap
corectrl         ~0.02 GiB swap

--- Totals ---
Total PSS accounted: 2.34 GiB
Total swap in use:   2.35 GiB
Kernel-reported used memory: 13.94 GiB
